<a href="https://colab.research.google.com/github/rabiakrann/Ileri-Oruntu-Tanima/blob/main/knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q kagglehub

In [6]:
import kagglehub
path = kagglehub.dataset_download(
    "muratkokludataset/dry-bean-dataset"
)
print("Dataset indirildi: ")
print(path)

Using Colab cache for faster access to the 'dry-bean-dataset' dataset.
Dataset indirildi: 
/kaggle/input/dry-bean-dataset


In [7]:
import glob

excel_files = glob.glob(
    path + "/**/*.xlsx",
    recursive=True
)

print(excel_files)

['/kaggle/input/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.xlsx']


In [8]:
import pandas as pd
import numpy as np
import random
import math
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.neighbors import KNeighborsClassifier

In [9]:
file_path = excel_files[0]

df = pd.read_excel(file_path)

df.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [10]:
print("Veri seti boyutu:", df.shape)
print("\nSütunlar:")
print(df.columns)

print("\nSınıf dağılımı:")
print(df["Class"].value_counts())

Veri seti boyutu: (13611, 17)

Sütunlar:
Index(['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength',
       'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent',
       'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2',
       'ShapeFactor3', 'ShapeFactor4', 'Class'],
      dtype='object')

Sınıf dağılımı:
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [11]:
x = df.drop("Class", axis=1).values.tolist()
y = df["Class"].tolist()

print("Örnek sayısı:", len(x))
print("Özellik sayısı:", len(x[0]))
print("İlk sınıf:", y[0])

Örnek sayısı: 13611
Özellik sayısı: 16
İlk sınıf: SEKER


In [12]:
import random

In [13]:
random.seed(42)

indices = list(range(len(x)))
random.shuffle(indices)
test_ratio = 0.20
test_size = int(len(x) * test_ratio)
test_indices = indices[:test_size]
train_indices = indices[test_size:]
x_train = [x[i] for i in train_indices]
y_train = [y[i] for i in train_indices]
x_test = [x[i] for i in test_indices]
y_test = [y[i] for i in test_indices]
print("Train örnek sayısı:", len(x_train))
print("Test örnek sayısı:", len(x_test))

Train örnek sayısı: 10889
Test örnek sayısı: 2722


In [14]:
import math
def calculate_mean_std(x):
  feature_count = len(x[0])
  means = []
  stds = []
  for j in range(feature_count):
    values = [row[j] for row in x]
    mean = sum(values)/len(values)

    variance=sum(
        (value-mean)**2
        for value in values
        ) /len(values)
    std = math.sqrt(variance)
    means.append(mean)
    stds.append(std)
  return means, stds

In [15]:
def standardize(x,means,stds):
  standardized = []
  for row in x:
    new_row=[]
    for j in range(len(row)):
      if stds[j]==0:
        new_row.append(0)
      else:
        value=(row[j]-means[j])/stds[j]
        new_row.append(value)
    standardized.append(new_row)
  return standardized

In [16]:
means, stds=calculate_mean_std(x_train)
x_train=standardize(x_train,means,stds)
x_test=standardize(x_test,means,stds)

In [17]:
import numpy as np
class KNN:
  def __init__(self, k=5):
    self.k=k
  def fit(self, x_train, y_train):
    self.x_train=np.asarray(x_train, dtype=float)
    self.y_train=np.asarray(y_train)
  def predict_one(self,x):
    distances=np.sqrt(((self.x_train-x)**2).sum(axis=1))
    nearest=np.argsort(distances)[:self.k]
    class_counts={}
    for label in self.y_train[nearest]:
      class_counts[label]=class_counts.get(label,0)+1
    return max(class_counts, key=class_counts.get)

  def predict(self, x_test):
     x_test=np.asarray(x_test, dtype=float)
     return np.array([self.predict_one(x) for x in x_test])


In [18]:
custom_knn = KNN(k=5)

custom_knn.fit(
    x_train,
    y_train
)
custom_predictions = custom_knn.predict(
    x_test
)

In [19]:
print("CUSTOM KNN SONUÇLARI\n")

print(
    classification_report(
        y_test,
        custom_predictions,
        zero_division=0
    )
)
custom_accuracy = accuracy_score(
    y_test,
    custom_predictions
)
print(
    "Custom KNN Accuracy:",
    custom_accuracy
)

CUSTOM KNN SONUÇLARI

              precision    recall  f1-score   support

    BARBUNYA       0.94      0.91      0.92       247
      BOMBAY       1.00      1.00      1.00       103
        CALI       0.95      0.94      0.95       332
    DERMASON       0.92      0.90      0.91       697
       HOROZ       0.96      0.96      0.96       416
       SEKER       0.95      0.96      0.95       409
        SIRA       0.84      0.87      0.85       518

    accuracy                           0.92      2722
   macro avg       0.94      0.93      0.94      2722
weighted avg       0.92      0.92      0.92      2722

Custom KNN Accuracy: 0.9224834680382072


In [20]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
sklearn_knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="euclidean",
    weights="uniform"
)

sklearn_knn.fit(x_train, y_train)
sklearn_predictions = sklearn_knn.predict(x_test)


In [21]:
print("SKLEARN KNN SONUÇLARI\n")
print(
    classification_report(
        y_test,
        sklearn_predictions,
        zero_division=0
    )
)
sklearn_accuracy = accuracy_score(
    y_test,
    sklearn_predictions
)
print(
    "Sklearn KNN Accuracy:",
    sklearn_accuracy
)

SKLEARN KNN SONUÇLARI

              precision    recall  f1-score   support

    BARBUNYA       0.93      0.91      0.92       247
      BOMBAY       1.00      1.00      1.00       103
        CALI       0.95      0.94      0.94       332
    DERMASON       0.91      0.91      0.91       697
       HOROZ       0.96      0.95      0.96       416
       SEKER       0.96      0.95      0.95       409
        SIRA       0.84      0.86      0.85       518

    accuracy                           0.92      2722
   macro avg       0.93      0.93      0.93      2722
weighted avg       0.92      0.92      0.92      2722

Sklearn KNN Accuracy: 0.9210139603232917


In [22]:
print(
    "Custom KNN Accuracy :",
    custom_accuracy
)
print(
    "Sklearn KNN Accuracy:",
    sklearn_accuracy
)

Custom KNN Accuracy : 0.9224834680382072
Sklearn KNN Accuracy: 0.9210139603232917


In [23]:
same_predictions = np.mean(
    custom_predictions == sklearn_predictions
)
print("Tahminlerin aynı olma oranı:",
    same_predictions)

Tahminlerin aynı olma oranı: 0.9944893460690669


In [24]:
different_count = np.sum(
    custom_predictions != sklearn_predictions
)
print(
    "Farklı tahmin edilen örnek sayısı:",
    different_count
)

Farklı tahmin edilen örnek sayısı: 15


In [25]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

In [26]:
X_cv = df.drop("Class", axis=1).values
y_cv = df["Class"].values

In [27]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
cv_accuracy_scores = []
cv_f1_scores = []

In [28]:
for fold, (train_index, test_index) in enumerate(
    skf.split(X_cv, y_cv),
    start=1
):
    x_train_fold = X_cv[train_index].tolist()
    x_test_fold = X_cv[test_index].tolist()

    y_train_fold = y_cv[train_index]
    y_test_fold = y_cv[test_index]
    means, stds = calculate_mean_std(x_train_fold)
    x_train_fold = standardize(
        x_train_fold,
        means,
        stds
    )
    x_test_fold = standardize(
        x_test_fold,
        means,
        stds
    )
    knn_cv = KNN(k=5)

    knn_cv.fit(
        x_train_fold,
        y_train_fold
    )
    predictions_cv = knn_cv.predict(
        x_test_fold
    )
    acc = accuracy_score(
        y_test_fold,
        predictions_cv
    )

    f1 = f1_score(
        y_test_fold,
        predictions_cv,
        average="macro"
    )

    cv_accuracy_scores.append(acc)
    cv_f1_scores.append(f1)

    print(
        f"Fold {fold}: "
        f"Accuracy = {acc:.4f}, "
        f"Macro F1 = {f1:.4f}"
    )

Fold 1: Accuracy = 0.9185, Macro F1 = 0.9318
Fold 2: Accuracy = 0.9313, Macro F1 = 0.9426
Fold 3: Accuracy = 0.9247, Macro F1 = 0.9390
Fold 4: Accuracy = 0.9280, Macro F1 = 0.9389
Fold 5: Accuracy = 0.9203, Macro F1 = 0.9320


In [29]:
cv_accuracy = np.mean(cv_accuracy_scores)
cv_f1 = np.mean(cv_f1_scores)

print("\n5-Fold Cross Validation")
print("Ortalama Accuracy:", cv_accuracy)
print("Ortalama Macro F1:", cv_f1)

print(
    "Accuracy Std:",
    np.std(cv_accuracy_scores)
)


5-Fold Cross Validation
Ortalama Accuracy: 0.9245467691202623
Ortalama Macro F1: 0.9368630862833808
Accuracy Std: 0.004746130575645314


In [30]:
custom_f1 = f1_score(
    y_test,
    custom_predictions,
    average="macro"
)

sklearn_f1 = f1_score(
    y_test,
    sklearn_predictions,
    average="macro"
)

In [31]:
print("1. Custom KNN")
print("Accuracy :", custom_accuracy)
print("Macro F1 :", custom_f1)

print("\n2. Sklearn KNN")
print("Accuracy :", sklearn_accuracy)
print("Macro F1 :", sklearn_f1)

print("\n3. Custom KNN + 5-Fold CV")
print("Accuracy :", cv_accuracy)
print("Macro F1 :", cv_f1)

1. Custom KNN
Accuracy : 0.9224834680382072
Macro F1 : 0.9352352544791716

2. Sklearn KNN
Accuracy : 0.9210139603232917
Macro F1 : 0.934041137874196

3. Custom KNN + 5-Fold CV
Accuracy : 0.9245467691202623
Macro F1 : 0.9368630862833808


In [32]:
results = pd.DataFrame({
    "Yöntem": [
        "Custom KNN",
        "Sklearn KNN",
        "Custom KNN + 5-Fold CV"
    ],

    "Accuracy": [
        custom_accuracy,
        sklearn_accuracy,
        cv_accuracy
    ],

    "Macro F1": [
        custom_f1,
        sklearn_f1,
        cv_f1
    ]
})

results

,Yöntem,Accuracy,Macro F1
0,Custom KNN,0.922483,0.935235
1,Sklearn KNN,0.921014,0.934041
2,Custom KNN + 5-Fold CV,0.924547,0.936863
